In [38]:
#Nombre Estudiante: Catalina Andrea Aguilera Letelier
#Generación: RTD-25-01-05-0019-1

#Consideraciones:
#Archivo: 03ventas_simuladas.csv (sep= ;)

#Librerías
from pyspark.sql import SparkSession
from datetime import datetime
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

#1 Preparar el dataset para entrenamiento del modelo
#Configuración programática
spark = SparkSession.builder \
    .appName("VentasSimuladas") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.instances", "3") \
    .getOrCreate()

#Crear un RDD base
sc = spark.sparkContext
rdd = sc.textFile("/content/03ventas_simuladas.csv")

#Separar cada línea por comas y estructurar cada registro como una tupla
rdd_data = rdd.map(lambda linea: tuple(linea.split(";")))

#Filtrar las filas incompletas o mal formateadas (por ejemplo, con campos vacíos o nulos)
rdd_limpio = rdd_data.filter(lambda x: all(i is not None and i != '' and i != 'invalid-date' for i in x))

#Convertir los valores numéricos (como cantidad y monto) a su tipo correspondiente (int o float)
header = rdd_limpio.first()
rdd_sin_header = rdd_limpio.filter(lambda row: row != header)
rdd_modificado = rdd_sin_header.map(lambda x: (x[0], x[1], int(x[2]), float(x[3]), float(x[4]), x[5]))

#Función para separar fecha y hora
def separar_fecha_hora(row):
    Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, ts_str = row

    # Convertir string a objeto datetime
    dt_obj = datetime.strptime(ts_str, '%Y-%m-%d %H:%M:%S')

    # Retornar (Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, Fecha)
    return (Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, dt_obj)

#Estandarizar el campo de fecha y hora
rdd_fecha = rdd_modificado.map(separar_fecha_hora)

#Función para separar fecha y hora
def franja_horaria(row):
    Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, Fecha = row

    #Franjas horarias
    if 6 <= Fecha.hour < 12: Franja = "Mañana"
    elif 12 <= Fecha.hour < 19: Franja = "Tarde"
    elif 19 <= Fecha.hour <=23 : Franja = "Noche"
    else: Franja = "Madrugada"

    # Retornar (Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, Fecha, Franja)
    return (Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, Fecha, Franja)

rdd_final = rdd_fecha.map(franja_horaria)

#Añadir una columna label binaria simulada (1: transacción riesgosa, 0: normal).
#Puedes asignarla aleatoriamente o con base en una regla simple (ej: monto_total > 7000 y venta en madrugada).
#Función transacción riesgosa
def transaccion_riesgosa(row):
  Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, Fecha, Franja = row
  if Monto_Total > 2000 and Franja == "Madrugada": Riesgo = 1
  else: Riesgo = 0

  return (Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, Fecha, Franja, Riesgo)

rdd_riesgo = rdd_final.map(transaccion_riesgosa)

#Aplicar StringIndexer y VectorAssembler para preparar las columnas en el formato requerido por MLlib: una columna features y una columna label.
df_spark = spark.createDataFrame(rdd_riesgo, ["Sucursal", "Producto", "Cantidad", "Precio_Unitario", "Monto_Total", "Fecha", "Franja", "Riesgo"])
indexers = [StringIndexer(inputCol=column, outputCol=column+"_index").fit(df_spark) for column in list(set(df_spark.columns)-set(["Cantidad","Precio_Unitario","Monto_Total","Fecha"]))]
pipeline = Pipeline(stages=indexers)
df_indexed = pipeline.fit(df_spark).transform(df_spark)
assembler = VectorAssembler(inputCols=["Cantidad", "Precio_Unitario", "Monto_Total", "Producto_index","Sucursal_index","Franja_index"], outputCol="features")
df_final = assembler.transform(df_indexed)
df_final.show()


+----------+--------+--------+---------------+-----------+-------------------+---------+------+--------------+------------+--------------+------------+--------------------+
|  Sucursal|Producto|Cantidad|Precio_Unitario|Monto_Total|              Fecha|   Franja|Riesgo|Producto_index|Riesgo_index|Sucursal_index|Franja_index|            features|
+----------+--------+--------+---------------+-----------+-------------------+---------+------+--------------+------------+--------------+------------+--------------------+
|Sucursal A|     Pan|       3|         989.78|    2969.34|2023-01-04 17:17:12|    Tarde|     0|           1.0|         0.0|           1.0|         0.0|[3.0,989.78,2969....|
|Sucursal A|  Aceite|       4|         563.57|    2254.28|2023-01-03 14:34:53|    Tarde|     0|           2.0|         0.0|           1.0|         0.0|[4.0,563.57,2254....|
|Sucursal B|Galletas|       4|         1180.5|     4722.0|2023-01-05 02:34:24|Madrugada|     1|           0.0|         1.0|           3

In [39]:
#2 Entrenar un modelo supervisado de clasificación con MLlib
#Entrena un modelo de clasificación binaria utilizando uno de los siguientes algoritmos de MLlib:
#LogisticRegression
#RandomForestClassifier
#DecisionTreeClassifier

modelo = RandomForestClassifier(labelCol="Riesgo_index", featuresCol="features", numTrees=5, seed=42)

#Debes:
#Dividir el dataset en entrenamiento y prueba (randomSplit).
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)
#Entrenar el modelo.
modelo_entrenado = modelo.fit(train_data)
#Generar predicciones sobre los datos de prueba.
predicciones = modelo_entrenado.transform(test_data)

In [40]:
#3 Evaluar el modelo y justificar decisiones técnicas
#Evalúa el desempeño del modelo utilizando al menos dos métricas de clasificación:
#accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="Riesgo_index", predictionCol="prediction", metricName="accuracy")
print("Precisión del modelo:", evaluator.evaluate(predicciones))

#areaUnderROC o F1-score
eval = BinaryClassificationEvaluator(labelCol="Riesgo_index", metricName="areaUnderROC")
print("AUC:", eval.evaluate(predicciones))

#Además, justifica por qué elegiste ese modelo, qué resultados obtuviste y cómo podrías mejorarlo en una versión futura
#(ej: validación cruzada, selección de variables, ajuste de hiperparámetros).
#Se realizaron pruebas con todas las variables "_index", la variable con mejores resultados fue: "Riesgo_index"
#con un 93% de precisión y un AUC de 0.97, números tan altos dan para pensar que existe un desbalance en la variable Riesgo.
#Para mejorar versiones futuras de este análisis, se recomienda realizar una validación cruzada con CrossValidator,
#ajustando hiperparámetros con ParamGridBuilder, como son numTrees y maxDepth.

Precisión del modelo: 0.9393939393939394
AUC: 0.9741379310344828
